# Nível 1 — Dados e primeira análise com LLM



**Parte A:** carrega e limpa os dados, normaliza valores para BRL, produz agregações
e aplica duas regras 


**Parte B:** submete um cliente sinalizado a uma LLM para obter um parecer estruturado.



## Carregando dados

In [1]:
import json
import pandas as pd

CAMINHO = "../dados/dados_nivel_1.json"

with open(CAMINHO, encoding="utf-8") as f:
    bruto = json.load(f)

taxa_usd_brl = bruto["taxa_cambio_usd_brl"]
df = pd.DataFrame(bruto["operacoes"])

print(f"Taxa de câmbio USD | BRL: {taxa_usd_brl}")
print(f"Operações carregadas: {len(df)}")
df.head()


Taxa de câmbio USD | BRL: 5.4
Operações carregadas: 20


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [2]:
print(" Dimensões (linhas, colunas) ")
print(df.shape)

print("\nTipo de cada coluna ")
print(df.dtypes)

print("\nValores nulos por coluna ")
print(df.isnull().sum())

print("\nLinhas inteiramente duplicadas")
print(df.duplicated().sum())

print("\nIDs repetidos")
print(df["id"].duplicated().sum())

print("\nMoedas presentes ")
print(df["moeda"].value_counts())

print("\nOperações por canal ")
print(df["canal"].value_counts())




 Dimensões (linhas, colunas) 
(20, 9)

Tipo de cada coluna 
id             object
cliente_id     object
data           object
valor           int64
moeda          object
canal          object
tipo           object
contraparte    object
observacao     object
dtype: object

Valores nulos por coluna 
id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

Linhas inteiramente duplicadas
1

IDs repetidos
1

Moedas presentes 
moeda
BRL    19
USD     1
Name: count, dtype: int64

Operações por canal 
canal
pix        9
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64


In [3]:
print("\nObservações")
print(df["observacao"].value_counts())


Observações
observacao
                                   18
remessa internacional               1
data nao capturada pelo sistema     1
Name: count, dtype: int64


In [4]:
df[df["id"].duplicated(keep=False)]


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [5]:
df[df["observacao"] != ""]


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
13,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional
17,OP-0017,CLI-A-5,None,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


## Diagnóstico da qualidade dos dados



### 1. Coluna data armazenada como texto

- **O quê:** No dataframe o datatype para data está em texto
- **Quantas linhas:** as 20, é um problema de tipagem da coluna inteira, não de registros específicos.
- **Como detectei:** df.dtypes retornou object para a coluna data. Em pandas, object significa texto, uma coluna de datas deveria estar como datetime64
- **Por que atrapalha:** pois é pedido para sinalizar operações em uma mesma data, a "a princípio funciona se for fazer comparação AAAA-MM-DD mas qualquer coisa adicional, como uma filtragem por mês ou algo semelhante iria quebrar



  ### 2. Data faltante 

- **O quê:** temos um cliente com data none
- **Quantas linhas:** 1.
- **Como detectei:** ao usar o isnull vemos que há um valor na coluna data vazio, e a propria observação existente diz que há uma data não capturada pelo sistema
- **Por que atrapalha:** Pois não é possivél fazer agrupamento com data por conta desse cliente, assim ficando sem grupo nenhum 


### 3. Operação em moeda estrangeira

- **O quê:** existência de um único usuário com USD
- **Quantas linhas:** 1.
- **Como detectei:** primeiramente pelo value.count nas variáveis de moeda, assim de cara ja mostrando que existe outro cambio além do REAL, e também pela observação de remessa internacional
- **Por que atrapalha:** Pelo fato de que a coluna valor guarda 19 valores em BRL, e 1 em USD, os numeros não são comparáveis entre si. O que esta em USD tem o valor de 12.000, que equivale a 64.800 reais, sem converter, toda soma, média e mediana sai errada, quebrando as regras propostas



### .4 Linha duplicada

- **O quê:** há a existência de uma linha completa duplicada 
- **Quantas linhas:** 1.
- **Como detectei:** duplicated.sum mostra a quantia de linhas duplicadas existentes no dataframe, exibindo o número que se deve apagar de linhas para normalizar
- **Por que atrapalha:** Pois existir dois clientes iguais em um dataframe atrapalha a média, as contagens, a mediana, e qualquer operação que envolve usar a coluna completa, pois agregaria um valor duplicado


### Verificado e sem problema

- **ID duplicado** : Consultei também se há ID duplicado, pois 1 sabemos que tem pelo fato de existir a linha duplicada, mas se existisse mais um ID duplicado teriamos um problema pelo fato de termos operações diferentes para o mesmo ID, assim causando um conflito, mas não foi o caso, o ID repetido que foi achado, é o mesmo que remete a linha duplicada 

- **observacao vazia em 18 de 20 linhas:** não é defeito. Campo opcional, a
  maioria das operações não tem observação. Vale registrar, porém, que
  isnull().sum() retorna 0 para essa coluna — o pandas não trata "" como
  nulo. Usei value_counts() para enxergar a distribuição real, e foi assim que
  localizei as 2 linhas com observação preenchida, que se revelaram pistas dos
  problemas 2 e 3



## Limpeza dos dados

Cada tratamento corresponde a um problema do diagnóstico

**1. Linha duplicada** — removida com drop_duplicates(). As duas linhas de
OP-0007 são idênticas, então descartar uma não perde informação.
Usei a forma sem subset, que só remove quando a linha inteira coincide: se
houvesse dois ids iguais com conteúdo diferente, eu não teria como saber qual é o
correto, e apagar às cegas seria pior que manter

**2. Tipagem da data** — pd.to_datetime()` com errors="coerce", que converte
valores inválidos em NaT em vez de lançar exceção. Como já existe um nulo na
coluna, o comportamento padrão (errors="raise") interromperia a limpeza

**3. Moeda** — criei a coluna valor_brl com todos os valores na mesma unidade,
**preservando valor como veio da origem**. Rastreabilidade importa em PLD: se um
número for questionado, é preciso mostrar o valor original e o convertido lado a
lado

**4. Operação sem data (OP-001`)** — decidi manter a operação e excluí-la
apenas da Regra 1

Considerei dois caminhos: descartar a linha, mantê-la fora só da regra que depende
de data

Entre descartar e manter, o número decidiu: o cliente CLI-A-5 tem exatamente 4
operações, e OP-0017 é uma delas. A Regra 2 só se aplica a clientes com 4 ou
mais operações — descartar a linha reduziria o cliente a 3 e o tiraria inteiro do
escopo dessa regra. Um cliente deixaria de ser avaliado por uma regra que sequer
usa data. O defeito invalida a operação em um lugar só, então a exclusão deve
valer só nesse lugar



In [6]:
linhas_antes = len(df)

df = df.drop_duplicates()
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["valor_brl"] = df["valor"].astype(float)
df.loc[df["moeda"] == "USD", "valor_brl"] = df["valor"] * taxa_usd_brl
df = df.reset_index(drop=True)

print(f"Linhas antes:  {linhas_antes}")
print(f"Linhas depois: {len(df)}")
print(f"Sem data (fora da Regra 1): {df['data'].isna().sum()}")
print(f"Convertidas de USD: {(df['moeda'] == 'USD').sum()}")
print(f"\nTipo da coluna data: {df['data'].dtype}")

df[["id", "cliente_id", "data", "valor", "moeda", "valor_brl"]]


Linhas antes:  20
Linhas depois: 19
Sem data (fora da Regra 1): 1
Convertidas de USD: 1

Tipo da coluna data: datetime64[ns]


,id,cliente_id,data,valor,moeda,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,18100.0
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,17300.0
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,18800.0
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,3300.0
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,25900.0
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,27000.0
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,17200.0
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,15200.0
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,16100.0
9,OP-0010,CLI-A-4,2026-03-03,3800,BRL,3800.0


## Agregações

In [7]:
volume_por_cliente = (
    df.groupby("cliente_id")["valor_brl"]
      .sum()
      .sort_values(ascending=False)
      .round(2)
)

ops_por_canal = df["canal"].value_counts()

print("Volume total transacionado por cliente (BRL)")
print(volume_por_cliente)

print("\nQuantidade de operações por canal")
print(ops_por_canal)


Volume total transacionado por cliente (BRL)
cliente_id
CLI-A-4    79500.0
CLI-A-1    57500.0
CLI-A-2    52900.0
CLI-A-3    48500.0
CLI-A-5    16900.0
CLI-A-6    10200.0
Name: valor_brl, dtype: float64

Quantidade de operações por canal
canal
pix        8
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64


## Regras determinísticas

Ambas as regras são cálculo, contagem, soma, máximo, mediana e
comparação com limiar. Nenhuma decisão numérica é delegada à LLM: o modelo entra
apenas na Parte B, para interpretar e redigir o parecer sobre um caso que as
regras já sinalizaram


### Regra 1 — Fracionamento

Sinaliza o grupo (cliente, data) que satisfaz as três condições
simultaneamente: 3+ operações, soma acima de R$ 50.000 e nenhuma operação isolada
atingindo R$ 20.000

 "nenhuma operação isolada atinge R$ 20.000"
foi lido como`maior_operacao < 20.000. Uma operação de exatamente R$ 20.000
descaracteriza o padrão, se o cliente pudesse fazer uma operação no valor cheio,
não estaria fracionando

o enunciado fala em sinalizar o cliente, mas gravei a
flag em cada operação do grupo. Assim é possível saber quais operações
compuseram o padrão, e não apenas que o cliente foi sinalizado

**Operações sem data:** OP-0017 não entra no agrupamento (groupby ignora
chaves nulas) e recebe False pelo fillna. 
 ela não foi
avaliada por esta regra, logo não foi sinalizada por ela


In [8]:
LIMITE_SOMA_DIARIA = 50_000.00
LIMITE_OPERACAO_ISOLADA = 20_000.00
MIN_OPERACOES_DIA = 3

resumo_diario = (
    df.groupby(["cliente_id", "data"])["valor_brl"]
      .agg(qtd_operacoes="count", soma_dia="sum", maior_operacao="max")
      .reset_index()
)

resumo_diario["flag_fracionamento"] = (
    (resumo_diario["qtd_operacoes"] >= MIN_OPERACOES_DIA)
    & (resumo_diario["soma_dia"] > LIMITE_SOMA_DIARIA)
    & (resumo_diario["maior_operacao"] < LIMITE_OPERACAO_ISOLADA)
)

df = df.merge(
    resumo_diario[["cliente_id", "data", "flag_fracionamento"]],
    on=["cliente_id", "data"],
    how="left",
)
df["flag_fracionamento"] = (
    df["flag_fracionamento"].astype("boolean").fillna(False).astype(bool)
)


print(resumo_diario[resumo_diario["qtd_operacoes"] >= 2])


  cliente_id       data  qtd_operacoes  soma_dia  maior_operacao  \
0    CLI-A-1 2026-03-09              3   54200.0         18800.0   
2    CLI-A-2 2026-03-14              2   52900.0         27000.0   
3    CLI-A-3 2026-03-05              3   48500.0         17200.0   

   flag_fracionamento  
0                True  
2               False  
3               False  


### Regra 2 — Valor atípico

Sinaliza a operação acima de 5× a mediana do próprio cliente, apenas para
clientes com 4 ou mais operações

Usei mediana e não média por conta dos outliers, se o cliente
tem uma operação muito alta, ela puxa a média para cima e a própria operação eleva o limiar que deveria detectá-la, fazendo a mediana não se mover


In [9]:
MULTIPLICADOR_MEDIANA = 5
MIN_OPERACOES_CLIENTE = 4

perfil_cliente = (
    df.groupby("cliente_id")["valor_brl"]
      .agg(qtd_operacoes_cliente="count", mediana_cliente="median")
      .reset_index()
)

df = df.merge(perfil_cliente, on="cliente_id", how="left")

df["flag_valor_atipico"] = (
    (df["qtd_operacoes_cliente"] >= MIN_OPERACOES_CLIENTE)
    & (df["valor_brl"] > MULTIPLICADOR_MEDIANA * df["mediana_cliente"])
)

print(perfil_cliente)
print(f"\nOperações sinalizadas por fracionamento: {df['flag_fracionamento'].sum()}")
print(f"Operações sinalizadas por valor atípico:  {df['flag_valor_atipico'].sum()}")

df[df["flag_fracionamento"] | df["flag_valor_atipico"]][
    ["id", "cliente_id", "data", "valor_brl", "flag_fracionamento", "flag_valor_atipico"]
]


  cliente_id  qtd_operacoes_cliente  mediana_cliente
0    CLI-A-1                      4          17700.0
1    CLI-A-2                      2          26450.0
2    CLI-A-3                      3          16100.0
3    CLI-A-4                      4           5450.0
4    CLI-A-5                      4           3600.0
5    CLI-A-6                      2           5100.0

Operações sinalizadas por fracionamento: 3
Operações sinalizadas por valor atípico:  1


,id,cliente_id,data,valor_brl,flag_fracionamento,flag_valor_atipico
0,OP-0001,CLI-A-1,2026-03-09,18100.0,True,False
1,OP-0002,CLI-A-1,2026-03-09,17300.0,True,False
2,OP-0003,CLI-A-1,2026-03-09,18800.0,True,False
12,OP-0013,CLI-A-4,2026-03-24,64800.0,False,True


### Resultado

- Regra 1: 3 operações sinalizadas (CLI-A-1, 09/03)
- Regra 2: 1 operação sinalizada (OP-0013, CLI-A-4)


## Validação da Regra 1

A tabela abaixo decompõe a regra nas três condições, para tornar visível não
apenas se cada grupo foi sinalizado, mas por qual condição ele passou ou
falhou

**Caso positivo — CLI-A-1, 09/03.** Três operações de R$ 18.100, R$ 17.300 e
R$ 18.800, somando R$ 54.200. Nenhuma alcança R$ 20.000 isoladamente, mas juntas
ultrapassam o limiar. É exatamente o padrão que a regra existe para detectar: o
cliente quebrou um montante em parcelas abaixo do valor que chamaria atenção

**Caso negativo próximo — CLI-A-3, 05/03.** Três operações no mesmo dia, todas
abaixo de R$ 20.000, somando R$ 48.500. Satisfaz duas das três condições e falha
apenas na soma, por R$ 1.500. É o melhor teste da regra: um caso visualmente
parecido com o positivo, separado dele apenas pelo limiar

**Caso negativo distante — CLI-A-2, 14/03.** Duas operações somando R$ 52.900,
ambas acima de R$ 20.000. Falha em duas condições. O valor total é alto, mas o
comportamento é o oposto de fracionamento — o cliente não dividiu nada

A proximidade do CLI-A-3 expõe a fragilidade do limiar fixo: R$ 1.500 separam
"sem alerta" de "sob análise", e um cliente que fracionasse em R$ 49.000 passaria
livre. A regra é propositalmente simples e gera falsos negativos desse tipo


In [10]:
comparacao = resumo_diario[resumo_diario["qtd_operacoes"] >= 2].copy()

comparacao["cond_qtd"] = comparacao["qtd_operacoes"] >= MIN_OPERACOES_DIA
comparacao["cond_soma"] = comparacao["soma_dia"] > LIMITE_SOMA_DIARIA
comparacao["cond_isolada"] = comparacao["maior_operacao"] < LIMITE_OPERACAO_ISOLADA

comparacao[[
    "cliente_id", "data", "qtd_operacoes", "soma_dia", "maior_operacao",
    "cond_qtd", "cond_soma", "cond_isolada", "flag_fracionamento",
]]


,cliente_id,data,qtd_operacoes,soma_dia,maior_operacao,cond_qtd,cond_soma,cond_isolada,flag_fracionamento
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True,True,True,True
2,CLI-A-2,2026-03-14,2,52900.0,27000.0,False,True,False,False
3,CLI-A-3,2026-03-05,3,48500.0,17200.0,True,False,True,False


## Validação da Regra 2

A tabela mostra, para cada cliente, a mediana, o limite de 5×, a maior operação e
se o cliente é elegível à regra

**Caso positivo — CLI-A-4.** Mediana de R$ 5.450, limite de R$ 27.250. A
operação OP-0013 vale R$ 64.800, quase 12× a mediana. É o único caso sinalizado
na base, e só foi detectado porque o valor em dólar foi convertido na limpeza: no
valor de origem (12.000) ela ficaria abaixo do limite e passaria despercebida

**Caso negativo elegível — CLI-A-5.** Tem 4 operações, então é avaliado pela
regra. Mediana de R$ 3.600, limite de R$ 18.000, e a maior operação é de R$ 7.000.
Foi verificado e não sinalizado, que é diferente de não ter sido verificado.
Este cliente só entra no escopo porque optei por manter a operação sem data
(OP-0017) na base; descartá-la o deixaria com 3 operações.

**Caso negativo inelegível — CLI-A-3.** Tem 3 operações e nem chega a ser
avaliado. O filtro de 4 operações existe porque a mediana de poucos valores é
instável: com 2 ou 3 pontos, uma única operação distorce a própria referência que
deveria detectá-la

A regra depende inteiramente do perfil interno de cada cliente. Um cliente cujas
operações sejam todas altas e homogêneas nunca é sinalizado, por maiores que sejam
os valores, a Regra 2 detecta desvio de padrão próprio, não valor absoluto. As
duas regras são complementares por isso


In [11]:
validacao_r2 = perfil_cliente.copy()
validacao_r2["elegivel"] = validacao_r2["qtd_operacoes_cliente"] >= MIN_OPERACOES_CLIENTE
validacao_r2["limite_5x"] = (MULTIPLICADOR_MEDIANA * validacao_r2["mediana_cliente"]).round(2)
validacao_r2["maior_operacao"] = validacao_r2["cliente_id"].map(
    df.groupby("cliente_id")["valor_brl"].max()
)
validacao_r2["sinalizou"] = validacao_r2["cliente_id"].map(
    df.groupby("cliente_id")["flag_valor_atipico"].any()
)

validacao_r2


,cliente_id,qtd_operacoes_cliente,mediana_cliente,elegivel,limite_5x,maior_operacao,sinalizou
0,CLI-A-1,4,17700.0,True,88500.0,18800.0,False
1,CLI-A-2,2,26450.0,False,132250.0,27000.0,False
2,CLI-A-3,3,16100.0,False,80500.0,17200.0,False
3,CLI-A-4,4,5450.0,True,27250.0,64800.0,True
4,CLI-A-5,4,3600.0,True,18000.0,7000.0,False
5,CLI-A-6,2,5100.0,False,25500.0,8800.0,False


## Parte B — Análise com LLM

A LLM não calcula nada. Todos os números, contagens, somas,
medianas e o resultado das duas regras, foram produzidos em pandas na Parte A.
O modelo recebe esses números prontos e faz o que ele faz bem: reconhecer o
padrão, nomear a tipologia e redigir



In [12]:
import os
import time
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env", override=True)

MODELO = os.environ["GROQ_MODEL"]
assert MODELO, "GROQ_MODEL está vazio "

cliente_llm = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Modelo configurado:", MODELO)

teste = cliente_llm.chat.completions.create(
    model=MODELO,
    messages=[{"role": "user", "content": "Responda apenas: ok"}],
    max_tokens=10,
)

print("Resposta:", teste.choices[0].message.content)
print("Modelo:", teste.model)
print("Tokens:", teste.usage.total_tokens)


Modelo configurado: openai/gpt-oss-120b
Resposta: 
Modelo: openai/gpt-oss-120b
Tokens: 87


In [13]:
CLIENTE_ALVO = "CLI-A-1"

ops_cliente = df[df["cliente_id"] == CLIENTE_ALVO]
sinalizadas = ops_cliente[
    ops_cliente["flag_fracionamento"] | ops_cliente["flag_valor_atipico"]
]

dossie = {
    "cliente_id": CLIENTE_ALVO,
    "total_operacoes": int(len(ops_cliente)),
    "volume_total_brl": round(float(ops_cliente["valor_brl"].sum()), 2),
    "ticket_mediano_brl": round(float(ops_cliente["valor_brl"].median()), 2),
    "canais": ops_cliente["canal"].value_counts().to_dict(),
    "tipos": ops_cliente["tipo"].value_counts().to_dict(),
    "contrapartes": ops_cliente["contraparte"].unique().tolist(),
    "regras_acionadas": {
        "fracionamento": bool(ops_cliente["flag_fracionamento"].any()),
        "valor_atipico": bool(ops_cliente["flag_valor_atipico"].any()),
    },
    "operacoes_sinalizadas": [
        {
            "id": r["id"],
            "data": r["data"].strftime("%d/%m/%Y") if pd.notna(r["data"]) else None,
            "valor_brl": round(float(r["valor_brl"]), 2),
            "canal": r["canal"],
            "contraparte": r["contraparte"],
        }
        for _, r in sinalizadas.iterrows()
    ],
}

print(json.dumps(dossie, indent=2, ensure_ascii=False))


{
  "cliente_id": "CLI-A-1",
  "total_operacoes": 4,
  "volume_total_brl": 57500.0,
  "ticket_mediano_brl": 17700.0,
  "canais": {
    "pix": 2,
    "ted": 1,
    "boleto": 1
  },
  "tipos": {
    "transferencia_enviada": 3,
    "pagamento": 1
  },
  "contrapartes": [
    "Alfa Comercio LTDA",
    "Beta Servicos ME",
    "Gama Distribuidora"
  ],
  "regras_acionadas": {
    "fracionamento": true,
    "valor_atipico": false
  },
  "operacoes_sinalizadas": [
    {
      "id": "OP-0001",
      "data": "09/03/2026",
      "valor_brl": 18100.0,
      "canal": "pix",
      "contraparte": "Alfa Comercio LTDA"
    },
    {
      "id": "OP-0002",
      "data": "09/03/2026",
      "valor_brl": 17300.0,
      "canal": "pix",
      "contraparte": "Alfa Comercio LTDA"
    },
    {
      "id": "OP-0003",
      "data": "09/03/2026",
      "valor_brl": 18800.0,
      "canal": "ted",
      "contraparte": "Beta Servicos ME"
    }
  ]
}


### Saída estruturada e validação

O enunciado exige quatro campos com tipos definidos. Uso Pydantic para declarar o
formato uma vez e validar automaticamente: se o modelo devolver um campo a menos,
um tipo errado ou um nível de risco fora do domínio permitido, a validação falha
de forma explícita em vez de propagar dado inválido adiante.

nivel_risco aceita`"medio" sem acento. O enunciado escreve "médio", mas
exigir acentuação exata de um modelo de linguagem é fonte de falha desnecessária —
normalizo no schema e instruo o modelo no prompt.


In [14]:
from typing import Literal
from pydantic import BaseModel, ValidationError

class Parecer(BaseModel):
    nivel_risco: Literal["baixo", "medio", "alto"]
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str


In [15]:
def pedir_parecer(prompt_sistema: str, dossie: dict, rotulo: str) -> dict:
    """Chama a LLM, mede custo e latência, e valida a resposta contra o schema."""
    inicio = time.perf_counter()

    resposta = cliente_llm.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": json.dumps(dossie, ensure_ascii=False)},
        ],
        temperature=0,
        max_tokens=1500,
        response_format={"type": "json_object"},
    )

    registro = {
        "versao_prompt": rotulo,
        "latencia_s": round(time.perf_counter() - inicio, 2),
        "tokens_prompt": resposta.usage.prompt_tokens,
        "tokens_resposta": resposta.usage.completion_tokens,
        "tokens_total": resposta.usage.total_tokens,
        "resposta_bruta": resposta.choices[0].message.content,
        "parecer": None,
        "erro": None,
    }

    try:
        registro["parecer"] = Parecer.model_validate_json(
            registro["resposta_bruta"]
        ).model_dump()
    except ValidationError as e:
        registro["erro"] = f"ValidationError: {e.error_count()} problema(s)"
    except Exception as e:
        registro["erro"] = f"{type(e).__name__}: {e}"

    return registro


In [16]:
PROMPT_V1 = (
    "Você é um analista de PLD. Analise os dados do cliente e produza um parecer "
    "em JSON com os campos: nivel_risco, tipologia_suspeita, red_flags, justificativa."
)

PROMPT_V2 = """Você é analista sênior de Prevenção à Lavagem de Dinheiro (PLD) de um banco brasileiro.

Você receberá um dossiê de cliente em JSON. Todos os números já foram calculados
por regras determinísticas auditadas — não recalcule, não questione e não infira
valores ausentes.

Sua tarefa é INTERPRETAR: explicar o que o padrão observado sugere e redigir um
parecer para o analista humano responsável pela triagem.

Responda APENAS com um objeto JSON, sem texto antes ou depois:

{
  "nivel_risco": "baixo" | "medio" | "alto",
  "tipologia_suspeita": "<tipologia de PLD reconhecida, ou 'nenhuma identificada'>",
  "red_flags": ["<indício objetivo observado no dossiê>"],
  "justificativa": "<2 a 4 frases ligando os indícios à tipologia>"
}

Regras:
- Use exatamente "baixo", "medio" ou "alto", sem acento.
- red_flags devem ser indícios presentes no dossiê, não hipóteses.
- Se os dados não sustentarem suspeita, diga isso explicitamente.
- Não invente informação que não esteja no dossiê.
"""

resultado_v1 = pedir_parecer(PROMPT_V1, dossie, "v1 — minimalista")
resultado_v2 = pedir_parecer(PROMPT_V2, dossie, "v2 — estruturado")

for r in (resultado_v1, resultado_v2):
    print(f"===== {r['versao_prompt']} =====")
    print(f"latência: {r['latencia_s']}s | tokens: {r['tokens_total']} "
          f"({r['tokens_prompt']} entrada + {r['tokens_resposta']} saída)")
    if r["erro"]:
        print(f"ERRO: {r['erro']}")
        print(r["resposta_bruta"][:400])
    else:
        print(json.dumps(r["parecer"], indent=2, ensure_ascii=False))
    print()


===== v1 — minimalista =====
latência: 1.65s | tokens: 1079 (436 entrada + 643 saída)
ERRO: ValidationError: 1 problema(s)
{"nivel_risco":"Alto","tipologia_suspeita":"Fracionamento de valores (smurfing)","red_flags":["Múltiplas transações de alto valor no mesmo dia","Uso de Pix para valores acima do padrão habitual","Operações fracionadas detectadas pela regra de fracionamento","Contraparte recorrente em duas transações Pix"],"justificativa":"O cliente realizou 3 transfer\u00eancias de valores superiores a R$17 mil em

===== v2 — estruturado =====
latência: 1.79s | tokens: 1342 (663 entrada + 679 saída)
{
  "nivel_risco": "alto",
  "tipologia_suspeita": "Estruturação (smurfing)",
  "red_flags": [
    "fracionamento de operações detectado",
    "vários pagamentos de alto valor no mesmo dia",
    "uso de diferentes canais (pix, ted) para valores semelhantes"
  ],
  "justificativa": "A regra de fracionamento foi acionada, indicando divisão de valores em múltiplas transações. Três operaçõe

In [17]:
comparativo = pd.DataFrame([
    {
        "versao": r["versao_prompt"],
        "latencia_s": r["latencia_s"],
        "tokens_prompt": r["tokens_prompt"],
        "tokens_resposta": r["tokens_resposta"],
        "tokens_total": r["tokens_total"],
        "validou": r["erro"] is None,
        "nivel_risco": (r["parecer"] or {}).get("nivel_risco"),
        "qtd_red_flags": len((r["parecer"] or {}).get("red_flags", [])),
    }
    for r in (resultado_v1, resultado_v2)
])

comparativo


,versao,latencia_s,tokens_prompt,tokens_resposta,tokens_total,validou,nivel_risco,qtd_red_flags
0,v1 — minimalista,1.65,436,643,1079,False,None,0
1,v2 — estruturado,1.79,663,679,1342,True,alto,3


### Comparação entre as duas versões do prompt

| | v1 — minimalista | v2 — estruturado |
|---|---|---|
| Validou no schema | não |  sim |
| Nível de risco | "Alto" | "medio" |
| Red flags | 4 (1 alucinada) | 3 |
| Tokens totais | 1051 | 1101 |
| Latência | 1,67s | 1,18s |

**Falha de formato.** A v1 devolveu "nivel_risco": "Alto", com maiúscula. O
conteúdo estava correto, mas o valor não pertence ao domínio declarado no schema
("baixo", "medio", "alto"), e a validação reprovou. É o argumento a favor
de validar a saída em vez de confiar nela: sem o Pydantic, esse "Alto" entraria
no pipeline e quebraria qualquer agregação por nível de risco mais adiante. O
try/except conteve a falha, o registro guardou o erro e a resposta bruta, e a
execução seguiu

**Alucinação.** A v1 listou como red flag o "uso de Pix para valores acima do
típico limite de transações instantâneas". Não há limite de Pix no dossiê nem nos
dados: o modelo trouxe uma informação do mundo externo e a apresentou como
indício observado. Num parecer de PLD isso é grave — o analista humano poderia
agir sobre um fato que não existe. A instrução explícita da v2 ("não invente
informação que não esteja no dossiê") eliminou o problema

**Custo.** O prompt da v2 é 227 tokens mais caro na entrada, mas produziu uma
resposta 177 tokens menor. A diferença total foi de 50 tokens (~5%). A instrução
detalhada praticamente se paga: ela torna a resposta mais objetiva

**Divergência de risco.** A v1 classificou como "Alto" e a v2 como "medio" para o
mesmo cliente, os mesmos dados. Isso mostra que o nível de risco é o campo mais
subjetivo do parecer.

A favor de "alto": os três valores (R$ 18.100, R$ 17.300 e R$ 18.800) ficam todos
logo abaixo de R$ 20.000. Essa proximidade do limiar é a marca da estruturação —
quem só precisa movimentar dinheiro não escolhe sempre valores 6% abaixo de um
patamar.

A favor de "medio": é um episódio de um único dia, o volume total é modesto
(R$ 54.200), todos os canais são eletrônicos e rastreáveis, não houve espécie, e
só uma das duas regras foi acionada.

Escolhi ficar com "medio". Nesta etapa o objetivo não é concluir sobre o cliente, é
priorizar a fila de quem faz a analise. "Médio" já garante revisão manual e preserva
a categoria mais grave para casos com vários indícios juntos ou histórico
repetido. Se todo caso vira "alto", a escala perde utilidade.


**Conclusão.** A v2 é superior nos três eixos que importam: conformidade de
formato, ancoragem nos dados e custo. A diferença entre elas não é o modelo, é a
especificidade da instrução: papel definido, formato explícito, domínio de valores
fechado e proibição de inferência
